# Core 01 - Tool

Objetivo: definir, tipar y ejecutar Tools mediante la API publica. Los inputs son simbolos configurables y los outputs se derivan de la instalacion.

**Lugar en el modelo:** `Tool` es la unidad determinista más pequeña. El alias `toolkit` nombra la caja completa `agentic_systems`; `ToolSet` nombra sólo una colección de Tools bajo un namespace.

**Evidencia exigida:** cada Tool debe validar su contrato y devolver un `RunResult` real con datos derivados de la ejecución.

**Límite de la evidencia:** agrupar Tools no crea un Agent, un loop ni un Provider.

## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_DEMO_SYMBOL | tool | Simbolo que la Tool inspecciona en toolkit.__all__. |
| schema | firma o Pydantic | Comparar las dos formas publicas de declarar Tools. |
| policy | ContractPolicySpec | Observar restricciones sin hardcodear ejecuciones. |

In [ ]:
import os

import agentic_systems as toolkit
from pydantic import BaseModel

SYMBOL = os.getenv("AGENTIC_SYSTEMS_DEMO_SYMBOL", "tool")

## 1) Decorador `toolkit.tool`

La funcion contiene logica de dominio. El decorador aporta schema, validacion, RunResult y tool events.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

quick_result = inspect_public_api.run({"symbol": SYMBOL})
assert quick_result.ok, quick_result.errors
assert quick_result.data["symbol"] == SYMBOL
toolkit.human_result(quick_result, title="Decorator Tool RunResult")

## 2) Tool con contratos Pydantic

El tipo de entrada y salida es parte del boundary observable.

In [ ]:
class SymbolInput(BaseModel):
    symbol: str

class SymbolOutput(BaseModel):
    symbol: str
    is_public: bool


def public_symbol(payload: SymbolInput) -> SymbolOutput:
    return SymbolOutput(
        symbol=payload.symbol,
        is_public=payload.symbol in toolkit.__all__,
    )

public_checker = toolkit.Tool(
    public_symbol,
    name="public_symbol",
    description="Verifica si un simbolo pertenece a la API publica.",
    input=SymbolInput,
    output=SymbolOutput,
)
typed_result = public_checker.run({"symbol": SYMBOL})
assert typed_result.ok, typed_result.errors
assert typed_result.data["symbol"] == SYMBOL
toolkit.human_result(typed_result, title="Typed Tool RunResult")

## 3) Validar contrato y presupuesto antes de ejecutar un Agent

In [ ]:
spec = toolkit.ContractPolicySpec(
    name="tutorial.inspect_once",
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=2),
)
check = spec.check(available_tools=[inspect_public_api.name, public_checker.name])
assert check.ok
toolkit.show_json({"spec": spec.describe(), "check": check.to_dict()}, title="ContractPolicySpec")

## 4) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.tool", "toolkit.Tool", "Tool.run", "toolkit.ContractPolicySpec",
    "toolkit.AgentContract", "toolkit.RunPolicy", "toolkit.human_result", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Tool API coverage")

## Resultado e interpretacion

Dos RunResult reales: uno desde decorador y otro desde contratos Pydantic.